# Create an Ensemble of Parameter Files


`meristem.paramgen` generates a set of parameter files for CLM or CLM-FATES from a spreadsheet of parameters and their ranges. It supports three sampling designs:

- **Latin Hypercube**: samples the parameter space evenly, so each parameter's range is well covered across the ensemble.
- **One-at-a-time (OAT)**: varies one parameter at a time between its minimum and maximum while holding all others at their defaults.
- **Morris**: generates trajectories through parameter space that change one parameter at a time, for Morris (elementary effects) global sensitivity analysis.

In this notebook we will build a one-at-a-time ensemble. The same workflow applies to the other designs, with a different `ensemble_type` and a few design-specific settings. For all of the options, including Latin Hypercube, Morris, grouping parameters, and posterior sampling, see `docs/paramgen.md`.

## Step 1: Investigate parameter ranges

The first step in a one-at-a-time sensitivity experiment is to select which parameters you want to modify and what minimum and maximum values to use for each. 
* Here we will start with three CLM-FATES parameters: 
  - `fates_leaf_stomatal_intercept` : *minimum unstressed stomatal conductance for Medlyn stomatal model* (μmol H<sub>2</sub>0 m<sup>-2</sup> s<sup>-1</sup>)
  - `fates_leaf_vcmax25top` : *maximum carboxylation rate of Rubisco at 25 °C, canopy top* (μmol CO<sub>2</sub> m<sup>-2</sup> s<sup>-1</sup>)
  - `fates_rad_leaf_clumping_index` : *self-occlusion of leaf scattering elements decreasing light interception* (unitless)



CLM and FATES provide suggested ranges for many parameters, but these ranges may not be appropriate for your ecosystem or experiment so you may modify them. In this experiment we will use the default minimum and maximum for each parameter. For more information about the suggested ranges see [Kennedy et al., 2025](https://doi.org/10.1029/2024MS004715) and [Foster et al., 2026]( https://doi.org/10.1029/2025MS005590). 

Open and inspect the default parameter ranges set in this file:

``meristem/examples/paramgen/fates_example/fates_param_dir/main.csv``

<div class="alert alert-info">
<strong>What are the minimum and maximum settings for the three parameters we've selected?</strong><br><br>
</div>

In [ ]:
from pathlib import Path
import pandas as pd
import json

# update this to your path
meristem_dir = Path("/Users/afoster/Documents/04_Code/meristem/")
meristem_dir.is_dir()

In [ ]:
csv_path = meristem_dir / "examples/paramgen/fates_example/fates_param_dir/main.csv"

param_df = pd.read_csv(csv_path, index_col=0)

for param_name in ('fates_leaf_stomatal_intercept', 'fates_leaf_vcmax25top', 'fates_rad_leaf_clumping_index'):
    print(param_name, "minimum = ", str(param_df.loc[param_name, 'param_min']))
    print(param_name, "maximum = ", str(param_df.loc[param_name, 'param_max']))
    print("----")

---
Did you notice that the `fates_leaf_vcmax25top` just says "pft"? This is because the ranges differ for each plant functional type. The parameter ranges per pft are in their own csv file:

In [ ]:
vcmax_path = meristem_dir / "examples/paramgen/fates_example/fates_param_dir/fates_leaf_vcmax25top.csv"
vcmax_pft_params = pd.read_csv(vcmax_path, index_col=0)
vcmax_pft_params

Each PFT has its own minimum and maximum for `fates_leaf_vcmax25top`, but in this example the PFTs are not varied independently. `fates_leaf_vcmax25top` is treated as a single parameter, so in the "minimum" ensemble member every PFT is set to its own minimum at the same time, and in the "maximum" member every PFT is set to its own maximum. That is why this parameter still contributes just two members (one minimum, one maximum) to the ensemble (see below).

If you wanted each PFT to vary on its own, you would set `expand_dim` to `fates_pft` for this parameter in `main.csv`. `paramgen` would then treat each PFT as a separate parameter, with its own minimum and maximum member. See the "Expand by index" section of `docs/paramgen.md` for details.

---

## Step 2: Generate parameter files

Now we are going to use ``meristem.paramgen`` to generate parameter files for our OAT sensitivity experiment. Given a set of parameters and their ranges, this module produces a collection of parameter files (NetCDF or FATES JSON) that can be fed directly into an ensemble of CLM/FATES model runs.

The ``meristem`` package has a built in sampling design for one-at-a-time minimum maximum sampling. We just have to set the ``ensemble_type`` and provide paths to our parameter information: 

In [ ]:
from meristem.paramgen import ParamEnsemble

fates_example_dir = meristem_dir / "examples/paramgen/fates_example"
out_dir = Path("/Users/afoster/Documents/paramfiles_OAT")

ensemble = ParamEnsemble.from_dict({
    "ensemble_type": "OAT",                                               # What type of sampling design to use? One-at-a-time (OAT) 
    "param_dir": fates_example_dir / "fates_param_dir",                   # Where are the csv files with the parameter information and values stored? 
    "default_param_file": fates_example_dir / "default_fates_param.json", # Where is the default fates parameter you want to start from? 
    "ensemble_dir": out_dir,                                              # Where do you want the generated parameter files to be stored?
    "file_prefix": "fates_oat",                                           # What should we call your experiment? This will be used as a prefix for the generated parameter files.
    "param_list": ["fates_leaf_stomatal_intercept","fates_leaf_vcmax25top", "fates_rad_leaf_clumping_index"], # which parameters do you want to modify? 
})
 
ensemble.create_ensemble()

You should see six parameter files, `fates_oat_0000.json` through `fates_oat_0005.json`. A one-at-a-time design varies one parameter at a time while holding all the others at their defaults, and it runs each parameter at its minimum and its maximum. With three parameters, that gives 3 × 2 = 6 ensemble members, each saved as its own parameter file.

### Check that the files were created correctly

Check that the files were created by listing the output directory:

In [ ]:
sorted([str(f) for f in out_dir.iterdir()])

You should see 6 parameter files, `fates_oat_000.json` through `fates_oat_0005.json`. Our one-at-a-time design varies one parameter at a time while holding all others at default, and runs each parameter at its minimum and maximum values. With three parameters, this gives 3 x 2 = 6 ensemble members, each saved as its own parameter files.

`paramgen` also writes three other files alongside the parameter files:

- `fates_oat_key.csv`: a "key" recording which parameter was changed in each file, and whether it was set to its minimum or maximum.
- `fates_oat.txt`: a plain text list of the ensemble member names, one per line. This is handy for scripting, such as looping over members when you set up model runs.
- `fates_oat_config.yaml`: metadata about the created parameter ensemble that can be used to reproduce your results

### Check out the parameter files

Now let's look inside the parameter files. Each file is a copy of the default parameter file with one parameter changed. Below we read the value of `fates_leaf_stomatal_intercept` from the default file and from every file in the ensemble.

In [ ]:
def get_param(path, name):
    """Read one parameter's entry from a FATES JSON parameter file."""
    with open(path) as f:
        return json.load(f)['parameters'][name]['data']

param_name = 'fates_leaf_stomatal_intercept'

print("default:", get_param(fates_example_dir / "default_fates_param.json", param_name))

for path in sorted(out_dir.glob("fates_oat_*.json")):
    print(path.name, get_param(path, param_name))

Most of the files match the default value, because in a one-at-a-time design each file only changes one parameter. Only two files differ for `fates_leaf_stomatal_intercept`: the one where it was set to its minimum, and the one where it was set to its maximum. Compare these values with the min and max you looked up in `main.csv` earlier. 

The key file can be used to understand what was done in the ensemble. It has one row per ensemble member, in the same order as the parameter files (row 0 corresponds to `fates_oat_0000.json`, etc.). For a one-at-a-time ensemble there is a `direction` column saying whether the perturbed parameter was set to its `minimum` or its `maximum`. All other parameters in that member were left at their default values.

Use the key to match the files you just looked at to their perturbations. Find the rows where `fates_leaf_stomatal_intercept` was changed, check their `direction`, and confirm that the minimum and maximum values you saw in the JSON files line up with those rows.

In [ ]:
key_path = out_dir / "fates_oat_key.csv"
param_key = pd.read_csv(key_path)
param_key

## Next steps

You now have a set of parameter files, one per ensemble member. The next notebook shows how to use `meristem.ensemble` to set up and submit model runs with them.

For more sampling designs and options, see `docs/paramgen.md`.